In [1]:
#Jupyter cell 1 — imports and connection
from rigol_dg1022 import RigolDG1022

RESOURCE = "USB0::0x1AB1::0x0642::DG1ZA231701902::INSTR"  # update if your VISA address differs
gen = RigolDG1022(RESOURCE)  # auto_open=True by default
print("Connected to:", gen.idn)


Connected to: Rigol Technologies,DG1022Z,DG1ZA231701902,03.01.12


### 1) Basic waveforms + outputs

#### Sin Function

In [ ]:

# 1 MHz sine on CH1, 2 Vpp, 0 V offset
freq=10_000_000

Vpi=5.97
coeff=.8



Vpp_Ch1=round(Vpi*coeff,3)

Vpp_Ch2=0.8




gen.set_waveform(ch=1, wave="SIN", freq_hz=freq, ampl_vpp=Vpp_Ch1, offset_v=0.0)

#Set phase of CH1
gen.set_phase_deg(ch=1, phase_deg=0) 

# 1 MHz sine on CH2 with 0° phase,  Vpp, 0 V offset
gen.set_waveform(ch=2, wave="SIN", freq_hz=freq, ampl_vpp=Vpp_Ch2, offset_v=0.04)

#Set phase of CH2
gen.set_phase_deg(ch=2, phase_deg=0)   # :SOURce2:PHAS 25 + *OPC? 

#Set ch1 and 2 impedance to 50 Ohms or INF

gen.set_load(ch=1, load='50')

gen.set_load(ch=2, load='50')


#### Switch Ch1 and Ch2 on

In [12]:
#Turn both outputs on
gen.output_on(1)
#gen.output_on(2)

#### Align Phases

In [ ]:
#Sync Phase
gen.sync_phase(ch=1)

#### Channels 1 & 2  off

In [11]:
#Turn both outputs off
gen.output_off(1)
#gen.output_off(2)

#### Pulse

In [ ]:

# Max PULSE frequency supported is 3 MHz on DG1022

ch=1
Vpi=5.7
coeff=0.8
Vpp_Ch=round(Vpi*coeff,2)

# Configure CH2 for PULSE waveform
gen.set_waveform(ch, "PULSE", freq_hz=3_000_000, ampl_vpp=Vpp_Ch, offset_v=0.0)

# Set output impedance to 50 Ohms
gen.set_load(ch=ch, load="50")

# Minimum pulse width is 20 ns (1 ns resolution)
gen.write("PULSe:WIDTh 50e-9")
gen.opc()

gen.output_on(ch)
print("Output: 3 MHz pulse, 20 ns width (hardware PULSE) on CH2")


#### Pulse Off

In [ ]:
gen.output_off(ch)

#### Close Connection

In [ ]:
gen.close()

### 2) Duty cycle (square) and ramp symmetry

In [ ]:

# IMPORTANT: APPLy:SQU/APPLy:RAMP resets duty/symmetry to 50% by design,
# so set the duty/symmetry AFTER you apply the waveform. [1](https://www.batronix.com/pdf/Rigol/ProgrammingGuide/DG1022_ProgrammingGuide_EN.pdf)

# CH1 square @ 10 kHz, then 20% duty
gen.set_waveform(ch=1, wave="SQU", freq_hz=10_000, ampl_vpp=2.0, offset_v=0.0)
gen.set_square_duty(ch=1, duty_percent=20.0)  # FUNCtion:SQUare:DCYCle {<percent>} [1](https://www.batronix.com/pdf/Rigol/ProgrammingGuide/DG1022_ProgrammingGuide_EN.pdf)

# CH2 ramp @ 5 kHz, then 70% symmetry
gen.set_waveform(ch=2, wave="RAMP", freq_hz=5_000, ampl_vpp=2.0, offset_v=0.0)
gen.set_ramp_symmetry(ch=2, symmetry_percent=70.0)  # FUNCtion:RAMP:SYMMetry {<percent>} [1](https://www.batronix.com/pdf/Rigol/ProgrammingGuide/DG1022_ProgrammingGuide_EN.pdf)


### 3) Burst setup (gated or triggered)

In [ ]:

# Example: TRIG burst on CH1, 5 cycles per trigger, internal trigger
gen.configure_burst(ch=1, enable=True, mode="TRIG", ncycles=5, trig_source="INT")
# Or: manual trigger (if trig_source="MAN")
gen.configure_burst(ch=1, enable=True, mode="TRIG", ncycles=3, trig_source="MAN")
gen.burst_trigger(ch=1)  # TRIGger:SINGle when source is MAN  [1](https://www.batronix.com/pdf/Rigol/ProgrammingGuide/DG1022_ProgrammingGuide_EN.pdf)

# Example: gated burst on CH2 (uses BURSt:MODE GATe and TRIGger:SOURce EXT/INT/MAN) [1](https://www.batronix.com/pdf/Rigol/ProgrammingGuide/DG1022_ProgrammingGuide_EN.pdf)
gen.configure_burst(ch=2, enable=True, mode="GATE", trig_source="EXT")


### 4) Frequency sweep

In [ ]:

# Linear sweep on CH1 from 1 kHz → 100 kHz in 2.5 s, internal trigger
gen.configure_sweep(ch=1, start_hz=1_000, stop_hz=100_000, time_s=2.5, spacing="LIN", trig_source="INT")
# Options include spacing LOG, direction UP/DOWN/UPDOWN depending on your method (SWEep:DIRection). [1](https://www.batronix.com/pdf/Rigol/ProgrammingGuide/DG1022_ProgrammingGuide_EN.pdf)


### 5) Upload an arbitrary waveform (points → VOLATILE → optional store)

In [ ]:

# Build a 1-cycle sine-like waveform with 1024 points, scaled to 14-bit [0..16383]
import math
N = 1024
points = [ int((math.sin(2*math.pi*i/N) * 0.5 + 0.5) * 16383) for i in range(N) ]

# Upload to VOLATILE, store as 'SINE1024', and select it on CH1
gen.upload_arb_points(ch=1, points=points, name="SINE1024", select_after=True)
# Then set USER/ARB frequency/amplitude/offset and enable output
gen.set_waveform(ch=1, wave="USER", freq_hz=10_000, ampl_vpp=2.0, offset_v=0.0)
gen.output_on(1)


### 6) Pulse generation

In [ ]:

from rigol_dg1022 import RigolDG1022

RESOURCE = "USB0::0x1AB1::0x0642::DG1ZA231701902::INSTR"

with RigolDG1022(RESOURCE) as gen:
    print("IDN:", gen.idn)
    ch = 1
    gen.set_load(ch, "50")

    # Max PULSE frequency supported is 3 MHz on DG1022
    gen.set_waveform(ch, "PULSE", freq_hz=3_000_000, ampl_vpp=2.0, offset_v=0.0)

    # Minimum pulse width is 20 ns (1 ns resolution)
    gen.write("PULSe:WIDTh 20e-9")
    gen.opc()

    gen.output_on(ch)
    print("Output: 3 MHz pulse, 20 ns width (hardware PULSE) on CH1")
